# FPL LSTM Points Predictor — Training Notebook

Trains a multi-input LSTM to predict a player's FPL points for the **next 3 gameweeks**
based on their last 5 appearances.

**Data source:** `fpl.gold.fact_gameweek_performance` (API + vaastav historical, 2016-present)  
**Model:** Registered to Unity Catalog as `fpl.gold.lstm_points_predictor`  
**Tracking:** MLflow experiment `/fpl/lstm_points_predictor`

Architecture mirrors the existing `lstm_train.py`:
- Sequence input (past 5 games × 19 features)
- Context input (static match context)
- Multi-output (next 3 gameweek point predictions)

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import joblib
import os

PAST_LEN   = 5   # look-back window
FUTURE_LEN = 3   # prediction horizon
TARGET_COL = 'total_points'
EPOCHS     = 50
BATCH_SIZE = 64

SEQUENCE_FEATURES = [
    'goals_scored', 'assists', 'saves', 'xG', 'xA', 'xGI', 'xGC',
    'influence', 'creativity', 'threat', 'ict_index',
    'minutes', 'clean_sheets', 'bonus', 'bps',
    'yellow_cards', 'red_cards', 'own_goals', 'penalties_missed',
]

CONTEXT_FEATURES = [
    'was_home', 'fixture_difficulty',
    'pos_GK', 'pos_DEF', 'pos_MID', 'pos_FWD',
]

MLFLOW_EXPERIMENT = os.getenv('MLFLOW_EXPERIMENT_NAME', '/fpl/lstm_points_predictor')
MODEL_NAME        = os.getenv('MODEL_NAME', 'fpl.gold.lstm_points_predictor')

## 1. Load Data from Gold Layer

In [ ]:
print('Loading fpl.gold.fact_gameweek_performance...')
df = (
    spark.table('fpl.gold.fact_gameweek_performance')
    .select(
        'player_name', 'position', 'team_name', 'season', 'gameweek',
        'kickoff_time', 'was_home', 'fixture_difficulty',
        TARGET_COL, *SEQUENCE_FEATURES,
    )
    .orderBy('player_name', 'kickoff_time')
    .toPandas()
)

print(f'Rows: {len(df):,}  |  Players: {df.player_name.nunique():,}  |  Seasons: {df.season.nunique()}')
df.head(3)

## 2. Feature Engineering

In [ ]:
# One-hot encode position
pos_dummies = pd.get_dummies(df['position'], prefix='pos')
for col in ['pos_GK', 'pos_DEF', 'pos_MID', 'pos_FWD']:
    df[col] = pos_dummies.get(col, 0).astype(int)

# Ensure was_home and fixture_difficulty are numeric
df['was_home']           = df['was_home'].astype(int)
df['fixture_difficulty'] = df['fixture_difficulty'].fillna(3).astype(float)

# Fill remaining NaN stat columns with 0
df[SEQUENCE_FEATURES + CONTEXT_FEATURES] = (
    df[SEQUENCE_FEATURES + CONTEXT_FEATURES].fillna(0)
)

print('Feature engineering complete.')

## 3. Fit Scalers and Create Sequences

In [ ]:
# Time-based 80/20 split (no leakage)
cutoff = df['kickoff_time'].quantile(0.8)
train_df = df[df['kickoff_time'] <= cutoff].copy()
test_df  = df[df['kickoff_time'] >  cutoff].copy()
print(f'Train: {len(train_df):,}  |  Test: {len(test_df):,}')

# Fit scalers on training data only
scaler_seq = MinMaxScaler()
scaler_ctx = MinMaxScaler()
scaler_y   = MinMaxScaler()

train_df[SEQUENCE_FEATURES] = scaler_seq.fit_transform(train_df[SEQUENCE_FEATURES])
train_df[CONTEXT_FEATURES]  = scaler_ctx.fit_transform(train_df[CONTEXT_FEATURES])
train_df[[TARGET_COL]]      = scaler_y.fit_transform(train_df[[TARGET_COL]])

test_df[SEQUENCE_FEATURES]  = scaler_seq.transform(test_df[SEQUENCE_FEATURES])
test_df[CONTEXT_FEATURES]   = scaler_ctx.transform(test_df[CONTEXT_FEATURES])
test_df[[TARGET_COL]]       = scaler_y.transform(test_df[[TARGET_COL]])


def create_sequences(data: pd.DataFrame, past: int, future: int):
    """Create (seq_input, ctx_input, y) arrays grouped by player."""
    X_seq, X_ctx, y = [], [], []
    for _, grp in data.groupby('player_name', sort=False):
        grp = grp.sort_values('kickoff_time')
        seq_vals = grp[SEQUENCE_FEATURES].values
        ctx_vals = grp[CONTEXT_FEATURES].values
        tgt_vals = grp[TARGET_COL].values
        for i in range(past, len(grp) - future + 1):
            X_seq.append(seq_vals[i - past:i])
            X_ctx.append(ctx_vals[i])
            y.append(tgt_vals[i:i + future])
    return np.array(X_seq), np.array(X_ctx), np.array(y)


X_seq_tr, X_ctx_tr, y_tr = create_sequences(train_df, PAST_LEN, FUTURE_LEN)
X_seq_te, X_ctx_te, y_te = create_sequences(test_df,  PAST_LEN, FUTURE_LEN)
print(f'Train sequences: {X_seq_tr.shape}  |  Test sequences: {X_seq_te.shape}')

## 4. Build Multi-Input LSTM Model

In [ ]:
def build_model(seq_len, n_seq_feat, n_ctx_feat, future_len):
    # Sequence branch
    seq_input = keras.Input(shape=(seq_len, n_seq_feat), name='sequence_input')
    x = layers.LSTM(64, return_sequences=True)(seq_input)
    x = layers.Dropout(0.2)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dropout(0.2)(x)

    # Context branch
    ctx_input = keras.Input(shape=(n_ctx_feat,), name='context_input')
    c = layers.Dense(16, activation='relu')(ctx_input)

    # Merge and output
    merged = layers.concatenate([x, c])
    merged = layers.Dense(32, activation='relu')(merged)
    output = layers.Dense(future_len, name='points_output')(merged)

    model = keras.Model(inputs=[seq_input, ctx_input], outputs=output)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    return model


model = build_model(
    seq_len=PAST_LEN,
    n_seq_feat=len(SEQUENCE_FEATURES),
    n_ctx_feat=len(CONTEXT_FEATURES),
    future_len=FUTURE_LEN,
)
model.summary()

## 5. Train with MLflow Tracking

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)
mlflow.tensorflow.autolog(log_models=False)  # manual model log below

with mlflow.start_run() as run:
    mlflow.log_params({
        'past_len':          PAST_LEN,
        'future_len':        FUTURE_LEN,
        'epochs':            EPOCHS,
        'batch_size':        BATCH_SIZE,
        'sequence_features': len(SEQUENCE_FEATURES),
        'context_features':  len(CONTEXT_FEATURES),
        'training_rows':     len(X_seq_tr),
        'data_source':       'fpl.gold.fact_gameweek_performance',
    })

    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True
    )

    history = model.fit(
        [X_seq_tr, X_ctx_tr], y_tr,
        validation_data=([X_seq_te, X_ctx_te], y_te),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[early_stop],
        verbose=1,
    )

    test_loss, test_mae = model.evaluate(
        [X_seq_te, X_ctx_te], y_te, verbose=0
    )
    mlflow.log_metrics({'test_loss': test_loss, 'test_mae': test_mae})
    print(f'Test MAE: {test_mae:.4f}')

    # Log scalers as artifacts
    joblib.dump(scaler_seq, '/tmp/scaler_seq.joblib')
    joblib.dump(scaler_ctx, '/tmp/scaler_ctx.joblib')
    joblib.dump(scaler_y,   '/tmp/scaler_y.joblib')
    mlflow.log_artifacts('/tmp', artifact_path='scalers')

    # Register model to Unity Catalog
    mlflow.tensorflow.log_model(
        model,
        artifact_path='lstm_model',
        registered_model_name=MODEL_NAME,
    )

    run_id = run.info.run_id
    print(f'Run ID: {run_id}')

## 6. Promote Champion Model

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Get the latest version just registered
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = max(int(v.version) for v in versions)

# Set 'champion' alias — used by the Model Serving endpoint and MCP agent
client.set_registered_model_alias(
    name=MODEL_NAME,
    alias='champion',
    version=str(latest_version),
)

print(f'Champion alias set → {MODEL_NAME} v{latest_version}')